# Fund Liquidity Stress and LMT Workflow

This notebook assesses the selected fund’s ability to meet redemption pressure at the valuation date and illustrates how selected liquidity management tools may respond under a dynamic redemption path.

The analysis separates three layers. First, it applies deterministic redemption stress scenarios sourced from the fund’s risk management policy. These scenarios represent point-in-time liability shocks, including investor concentration events such as the largest holder or three largest holders redeeming. Second, it compares each redemption shock with the fund’s asset-side liquidity profile, based on cash, liquid holdings and liquidation buckets. Third, it applies selected LMT mechanics, including gates, anti-dilution adjustments and suspension indicators, where redemption pressure exceeds defined policy thresholds.

The dynamic redemption path complements the point-in-time stress analysis. It shows how redemption pressure may evolve across future dealing dates after LMT mechanics are applied, including paid redemptions, deferred redemptions, backlog effects and conditional contagion after a prior gate event.

References for building this notebook can be found [here](../../docs/notebooks_notes/liquidity_references.md).


# Setup

In [ ]:
# connect to SQL database
from src.data.database   import get_engine
ENGINE  = get_engine()

# fund to be analysed
FUND = "UCITS_Balanced"

# In this repo all computation refer to the same date
from src.config import VALUATION_DATE
VALUATION_DATE

In [ ]:
# Query A risk-ready DataFrame with all columns needed for
# risk computtion including liquidty risk
from src.data.enrichment import get_risk_ready_df
risk_df = get_risk_ready_df(ENGINE, FUND, VALUATION_DATE)
NAV = risk_df['market_value_eur'].sum()

## 1. Fund Liquidity Terms and Policy Inputs

The fund profile and liquidity monitoring inputs below are queried from the electronic fund records.
> Note: <br>
> Pct ADV: reduces avg daily volume to estimate max amount the fund can trade on that ecuritu.
> Stress window is the monitoring horizon used to interpret how much liquidity is available under stress.


In [ ]:
# Display fund overview combined with liquidity monitoring parameters
import src.ui.liquidity_calibration_display as lcd
lcd.display_fund_liquidity_overview(
    fund_id=FUND,
    engine=ENGINE,
    export_id="01",
)

## 2. Portfolio Liquidity Profile

This section estimates how quickly the selected fund’s holdings could be converted into cash under the project’s liquidity classification. This creates the asset-side liquidity view used later in the stress test.


Days-to-liquidate estimates position-level liquidation timelines.
Liquidity buckets classify assets by expected settlement speed.

In [ ]:
# Liquidity profile — plot only
from src.ui.liquidity_plot import plot_liquidity_profile
from src.risk.risk_utils import compute_liquidity_profile

# Compute liquidity profile
liquidity_pct_adv = 0.25
liq = compute_liquidity_profile(risk_df, pct_adv=liquidity_pct_adv)
bucket_full = liq['bucket_full']
risk_df_liq = liq['risk_df_liq']

plot_liquidity_profile(bucket_full, FUND, metric='pct_nav_abs', valuation_date=VALUATION_DATE, export_id="02")


## 3. Investor Base and Redemption Scenario Inputs

This section loads the liability-side inputs used in the redemption stress workflow.

The investor register supports investor concentration checks. The RMP redemption scenarios define the point-in-time redemption shocks applied to the valuation-date NAV before LMT mechanics are considered.


In [ ]:
# Load investor registers and calibration inputs
from src.data.reference_data import load_investor_and_calibration_data

# Load calibration data for UCITS_Balanced
data = load_investor_and_calibration_data(FUND)
investor_inputs = data['investor_inputs']
calibration_inputs = data['calibration_inputs']
calibration_config = data['calibration_config']

# Display top 5 investors
lcd.display_top_investors(investor_inputs, FUND, VALUATION_DATE, top_n=5)


**Investor Redemption for Point-in_time**

The Risk Management Policy defines the redemption rate assumptions for each investor category under the applicable liquidity stress scenarios. These rates are loaded from the electronic data profile.

In [ ]:

# Compute redemption scenarios
from src.computation.liquidity_calibration import (
    compute_redemption_scenarios,
)
scenarios_result = compute_redemption_scenarios(investor_inputs, calibration_config)


## 4. Point-in-Time Redemption Stress

This section applies one selected redemption scenario to the fund’s valuation-date portfolio. It compares the redemption amount with the liquid assets available under the project liquidity buckets. The purpose is to test whether the fund could meet a defined redemption shock before applying any LMT mechanics.

In [ ]:
# Redemption stress — use existing function
import src.ui.print_html_utils as phtml
from src.data.reference_data import get_stress_testing_params
from src.pipeline.liquidity_policy import load_redemption_scenarios_from_rmp

# Load redemption scenarios from RMP
REDEMPTION_SCENARIOS = load_redemption_scenarios_from_rmp(FUND)

# Get stress testing params
params = get_stress_testing_params(FUND, calibration_inputs, redemption_scenario="Large")
notice_period_days = params['notice_days']

# Display using existing function
phtml.display_redemption_stress(
    FUND,
    notice_period_days,
    REDEMPTION_SCENARIOS,
    NAV,
    risk_df_liq,
    valuation_date=VALUATION_DATE,
    export_id="03",
)


## 5. Dynamic Redemption Path — Before LMT Effects

This section simulates the redemption path over 12 months without liquidity management tools applied. It shows how redemption pressure evolves month-by-month when no gates, swing pricing, or suspension mechanics are active.

The purpose is to establish a baseline: raw redemption demand, resulting fund-level outflows, and NAV evolution in an unmanaged scenario. This provides the counterfactual against which LMT effects can be measured.

The dynamic path uses the same monthly redemption schedule as the point-in-time stress scenarios but projects it forward across future dealing dates, allowing for backlog and contagion effects to accumulate.

In [ ]:
# Display LMT parameters from calibration
lcd.display_lmt_parameters(calibration_inputs, FUND, export_id="04")

In [ ]:
# LMT Trigger Analysis — Before and After LMT Effects
from src.pipeline.lmt_trigger_analysis import (
    build_lmt_analysis_inputs,
    prepare_scenarios_data,
)
from src.computation.liquidity import lmt_trigger_analysis
from src.data.enrichment import get_risk_ready_df
from IPython.display import display, HTML

# Build LMT inputs from calibration
lmt_inputs = build_lmt_analysis_inputs(
    FUND,
    calibration_inputs,
    calibration_config,
)

# Get NAV and liquid_pct for direct lmt_trigger_analysis calls
risk_df = get_risk_ready_df(ENGINE, FUND, VALUATION_DATE)
nav = risk_df['market_value_eur'].sum()
lmt_config = lmt_inputs['lmt_config']
liquid_pct = lmt_config['liquid_pct']
schedule = lmt_inputs['schedule']

# Scenario 1: Dynamic path WITHOUT LMT effects (all None to disable)
df_analysis_before_lmt = lmt_trigger_analysis(
    nav=nav,
    liquid_pct=liquid_pct,
    gate_threshold=None,
    swing_threshold=None,
    redemption_schedule=schedule,
    consecutive_gate_for_suspension=None,
    contagion_multiplier=1.0,
    apply_contagion=False,
)

# Prepare scenarios for display
scenarios_data = prepare_scenarios_data(scenarios_result)

# Display LMT parameters from calibration
lcd.display_lmt_parameters(calibration_inputs, FUND, export_id="04")

display(HTML("<h4>Before LMT (no gates, swing, or suspension; no contagion)</h4>"))
display(df_analysis_before_lmt[['month', 'paid_eur', 'deferred_eur', 'backlog_eur', 'gate_active', 'swing_active']].head())


In [ ]:
# Plots: Redemption path BEFORE LMT effects
lcd.plot_redemption(df_analysis_before_lmt, FUND, VALUATION_DATE, export_id="06a", scenario_label="Before LMT")
lcd.plot_lmt_nav_evolution(df_analysis_before_lmt, FUND, VALUATION_DATE, export_id="07a", scenario_label="Before LMT")

## 6. Dynamic Redemption Path — After LMT Effects

This section applies the fund's liquidity management tool settings (gates, swing pricing, suspension) to the same redemption path.

The comparison between Section 5 (before LMT) and this section (after LMT) shows the treatment effect: how much deferral is triggered, when suspension becomes active, and how redemption outflows change under policy constraints.

The after-LMT run includes conditional investor feedback. The feedback multiplier is applied only after a gate event and represents second-round redemption pressure following LMT activation. This contagion effect models the behavioural response of investors to deferral or suspension, which may trigger additional redemptions in subsequent months.

Swing pricing or anti-dilution adjustments reduce the amount paid to redeeming investors, protecting remaining investors from dilution caused by large outflows. Gates defer redemptions into a backlog when outflows exceed thresholds. Suspension halts redemptions entirely when gate and backlog conditions are simultaneously breached.

This section models a dynamic redemption path under liquidity pressure.

The objective is to show:

* the redemption demand generated by the investor-base assumptions;
* the amount paid immediately from available liquidity;
* the amount deferred when the redemption gate is triggered;
* the backlog created by unpaid or deferred redemptions;
* the effect of conditional contagion after LMT activation, where the redemption pressure increases following a prior gate event.

The after-LMT run therefore reflects both the direct application of the LMT mechanics and the second-round investor feedback triggered by the use of gates.


In [ ]:
# Scenario 2: Dynamic path WITH LMT effects (policy thresholds)
df_analysis_after_lmt = lmt_trigger_analysis(
    nav=nav,
    liquid_pct=liquid_pct,
    gate_threshold=lmt_config['gate_threshold'],
    swing_threshold=lmt_config['swing_threshold'],
    redemption_schedule=schedule,
    consecutive_gate_for_suspension=lmt_config['consec_gate'],
    contagion_multiplier=lmt_config['contagion'],
    apply_contagion=True,
)
display(HTML("<h4>After LMT (with policy thresholds and conditional contagion feedback)</h4>"))
display(df_analysis_after_lmt[['month', 'paid_eur', 'deferred_eur', 'backlog_eur', 'gate_active', 'swing_active']].head())

In [ ]:
len('post-gate contagion')

In [ ]:
# Plots: Redemption path AFTER LMT effects
lcd.plot_redemption(df_analysis_after_lmt, FUND, VALUATION_DATE, export_id="06b", scenario_label="After LMT", gate_threshold=lmt_config['gate_threshold'], swing_threshold=lmt_config['swing_threshold'], consecutive_gate_for_suspension=lmt_config['consec_gate'])
lcd.plot_lmt_nav_evolution(df_analysis_after_lmt, FUND, VALUATION_DATE, export_id="07b", scenario_label="After LMT")
lcd.plot_lmt_flags(df_analysis_after_lmt, FUND, VALUATION_DATE, export_id="08b")